In [9]:
import os

import numpy as np
import prettytable as pt
import pandas as pd
from scipy.optimize import curve_fit
from scipy.integrate import quad

import hist


Caching the list of root modules, please wait!
(This will only be done once - type '%rehashx' to reset cache!)



In [17]:
cats = {
    # 'VBFHH': {
    #     "is_VBFHH_sig": 0.774,
    # },
    'cat1': {
        "is_ggHH_sig": 0.20105975753991212,
        "is_nonRes_bkg": 0.00033920225777953755,
        "is_Res_bkg": 0.5155782175840005
    },
    'cat2': {
        "is_ggHH_sig": 0.8633095100474404,
        "is_nonRes_bkg": 0.0015707913189143552,
        "is_Res_bkg": 0.7641090808766726
    },
    'cat3': {
        "is_ggHH_sig": 0.7121864823883823,
        "is_nonRes_bkg": 0.006492482419944794,
        "is_Res_bkg": 0.5593729774038129
    }
}

eras = ['preEE', 'postEE', 'preBPix', 'postBPix', '2024']
dirpaths = [f"/eos/uscms/store/group/lpcdihiggsboost/tsievert/Version_20260129_FCPnb/individual_samples/{era}" for era in eras]
samples = os.listdir(dirpaths[0])
files = [os.path.join(dirpath, sample, "events.parquet") for sample in samples for dirpath in dirpaths]
files24 = [os.path.join(dirpaths[-1], sample, "events.parquet") for sample in samples]
files

['/eos/uscms/store/group/lpcdihiggsboost/tsievert/Version_20260129_FCPnb/individual_samples/preEE/BBHto2G_M_125/events.parquet',
 '/eos/uscms/store/group/lpcdihiggsboost/tsievert/Version_20260129_FCPnb/individual_samples/postEE/BBHto2G_M_125/events.parquet',
 '/eos/uscms/store/group/lpcdihiggsboost/tsievert/Version_20260129_FCPnb/individual_samples/preBPix/BBHto2G_M_125/events.parquet',
 '/eos/uscms/store/group/lpcdihiggsboost/tsievert/Version_20260129_FCPnb/individual_samples/postBPix/BBHto2G_M_125/events.parquet',
 '/eos/uscms/store/group/lpcdihiggsboost/tsievert/Version_20260129_FCPnb/individual_samples/2024/BBHto2G_M_125/events.parquet',
 '/eos/uscms/store/group/lpcdihiggsboost/tsievert/Version_20260129_FCPnb/individual_samples/preEE/DDQCDGJET/events.parquet',
 '/eos/uscms/store/group/lpcdihiggsboost/tsievert/Version_20260129_FCPnb/individual_samples/postEE/DDQCDGJET/events.parquet',
 '/eos/uscms/store/group/lpcdihiggsboost/tsievert/Version_20260129_FCPnb/individual_samples/preBPix

In [4]:
files24

['/eos/uscms/store/group/lpcdihiggsboost/tsievert/Version_20260129_FCPnb/individual_samples/2024/BBHto2G_M_125/events.parquet',
 '/eos/uscms/store/group/lpcdihiggsboost/tsievert/Version_20260129_FCPnb/individual_samples/2024/DDQCDGJET/events.parquet',
 '/eos/uscms/store/group/lpcdihiggsboost/tsievert/Version_20260129_FCPnb/individual_samples/2024/GGJets/events.parquet',
 '/eos/uscms/store/group/lpcdihiggsboost/tsievert/Version_20260129_FCPnb/individual_samples/2024/GluGluHToGG_M_125/events.parquet',
 '/eos/uscms/store/group/lpcdihiggsboost/tsievert/Version_20260129_FCPnb/individual_samples/2024/GluGlutoHHto2B2G_kl_1p00_kt_1p00_c2_0p00/events.parquet',
 '/eos/uscms/store/group/lpcdihiggsboost/tsievert/Version_20260129_FCPnb/individual_samples/2024/TTGG/events.parquet',
 '/eos/uscms/store/group/lpcdihiggsboost/tsievert/Version_20260129_FCPnb/individual_samples/2024/VBFHToGG_M_125/events.parquet',
 '/eos/uscms/store/group/lpcdihiggsboost/tsievert/Version_20260129_FCPnb/individual_samples/

In [5]:
test = pd.read_parquet(files[0])
test.columns

Index(['lead_seediEtaOriX', 'lead_cutBased', 'lead_electronVeto',
       'lead_hasConversionTracks', 'lead_isScEtaEB', 'lead_isScEtaEE',
       'lead_mvaID_WP80', 'lead_mvaID_WP90', 'lead_pixelSeed', 'lead_seedGain',
       ...
       'nonResReg_lead_bjet_btag_WP_T', 'nonResReg_lead_bjet_btag_WP_XT',
       'nonResReg_lead_bjet_btag_WP_XXT', 'nonResReg_sublead_bjet_btag_WP_L',
       'nonResReg_sublead_bjet_btag_WP_M', 'nonResReg_sublead_bjet_btag_WP_T',
       'nonResReg_sublead_bjet_btag_WP_XT',
       'nonResReg_sublead_bjet_btag_WP_XXT', 'rel_xsec_weight', 'weight_tot'],
      dtype='str', length=1290)

In [11]:
samples

['BBHto2G_M_125',
 'DDQCDGJET',
 'GGJets',
 'GluGluHToGG_M_125',
 'GluGlutoHHto2B2G_kl_1p00_kt_1p00_c2_0p00',
 'TTGG',
 'VBFHToGG_M_125',
 'VHtoGG_M_125',
 'ttHtoGG_M_125']

In [16]:
#############################################################
# ASCii histogram for rapid plotting
def ascii_hist(x, bins=10, weights=None):
    N,X = np.histogram(x, bins=bins, weights=weights)
    width = 50
    nmax = np.max(N.max())
    for (xi, n) in zip(X,N):
        bar = '#'*int(n*width/nmax)
        xi = '{0: <8.4g}'.format(xi).ljust(10)
        print('{0}| {1}'.format(xi,bar))
def ascii_hist(x, bins=10, weights=None, fit=None):
    N,X = np.histogram(x, bins=bins, weights=weights)
    width = 50
    nmax = np.max([N.max(), fit.max()])
    if fit is None:
        for (xi, n) in zip(X,N):
            bar = '#'*int(n*width/nmax)
            xi = '{0: <8.4g}'.format(xi).ljust(10)
            print('{0}| {1}'.format(xi,bar))
    else:
        for (xi, n, fiti) in zip(X,N,fit):
            bar = '#'*int(n*width/nmax)
            if fiti > n: bar = bar + ' '*(int(fiti*width/nmax) - int(n*width/nmax)) + '+'
            else: bar = ''.join([bar[j] if j != int(fiti*width/nmax) else '+' for j in range(len(bar))])
            xi = '{0: <8.4g}'.format(xi).ljust(10)
            print('{0}| {1}'.format(xi,bar))

#############################################################
# Sideband fit functions for nonRes bkg estimaton
def exp_func(x, a, b):
    return a * np.exp(b * x)
def sd_hist(mass: np.ndarray, weight: np.ndarray, fit_bins: list[float]):
    return hist.Hist(
        hist.axis.Regular(int((fit_bins[1]-fit_bins[0])//fit_bins[2]), fit_bins[0], fit_bins[1], name="var", growth=False, underflow=False, overflow=False), 
        storage='weight'
    ).fill(var=mass, weight=weight)
def exp_mass_fit(mass: np.ndarray, weight: np.ndarray, fit_bins: list[float], sigma: bool=False):
    _hist_ = sd_hist(mass, weight, fit_bins)
    params, _ = curve_fit(
        exp_func, _hist_.axes.centers[0]-_hist_.axes.centers[0][0], _hist_.values(), p0=(_hist_.values()[0], -0.1), 
        sigma=np.where(_hist_.values() != 0, np.sqrt(_hist_.variances()), 0.76) if sigma else None
    )
    print(_hist_)
    ascii_hist(mass, bins=np.arange(fit_bins[0], fit_bins[1], fit_bins[2]), weights=weight, fit=exp_func(_hist_.axes.centers[0]-_hist_.axes.centers[0][0], a=params[0], b=params[1]))
    return _hist_, params
def est_yield(mass: np.ndarray, weight: np.ndarray, fit_bins: list[float], sr_masscut: list[float], sigma: bool=False):
    _hist_, fit_params = exp_mass_fit(mass, weight, fit_bins, sigma=sigma)
    return quad(exp_func, sr_masscut[0]-_hist_.axes.centers[0][0], sr_masscut[1]-_hist_.axes.centers[0][0], args=tuple(fit_params))[0] / fit_bins[2]

In [20]:
field_names = ['Category'] + samples + ['nonRes SB fit', 's/b', 's/b with fit']
table = pt.PrettyTable(field_names=field_names, float_format=".3")

not_prev_cut_mask = {}
for name, cuts in cats.items():
    new_row = [name]
    nonRes_sideband = pd.DataFrame({'mass': pd.Series(dtype='float'), 'weight_tot': pd.Series(dtype='float')})

    for sample in samples:
        sample_sum = 0.

        for file in files:
            if sample not in file: continue

            file_df = pd.read_parquet(file, columns=['mass', 'weight_tot'])
            preds = np.load(file.replace('events.parquet', 'y.npy'))
            pred_fields = ["is_nonRes_bkg_score", "is_Res_bkg_score", "is_ggHH_sig_score", "is_VBFHH_sig_score"]
            for i, field in enumerate(pred_fields):
                file_df[field] = preds[:, i]
                
            if file not in not_prev_cut_mask: not_prev_cut_mask[file] = np.ones(len(file_df), dtype=bool)
            
            pass_cut_mask = not_prev_cut_mask[file]
            for cut_name, cut in cuts.items():
                pass_cut_mask = np.logical_and(
                    pass_cut_mask, file_df.loc[:, f'{cut_name}_score'].gt(cut).to_numpy() if cut_name.endswith('sig') else file_df.loc[:, f'{cut_name}_score'].lt(cut).to_numpy()
                )
            pass_cut_sr_mask = np.logical_and(
                pass_cut_mask,
                np.logical_and(file_df.loc[:, 'mass'].gt(122.5).to_numpy(), file_df.loc[:, 'mass'].lt(127.).to_numpy())
            )

            sample_sum += file_df.loc[pass_cut_sr_mask, 'weight_tot'].sum()

            if sample in ['DDQCDGJET', 'GGJets', 'TTGG']:
                pass_cut_sideband_mask = np.logical_and(
                    pass_cut_mask,
                    np.logical_or(file_df.loc[:, 'mass'].lt(120.).to_numpy(), file_df.loc[:, 'mass'].gt(130.).to_numpy())
                )
                nonRes_sideband = pd.concat([nonRes_sideband, file_df.loc[pass_cut_sideband_mask, ['mass', 'weight_tot']]], ignore_index=True)

            not_prev_cut_mask[file] = np.logical_and(not_prev_cut_mask[file], ~pass_cut_mask)

        new_row.append(sample_sum)

    sb_est_yield = est_yield(nonRes_sideband['mass'], nonRes_sideband['weight_tot'], [100., 180., 5.], [122.5, 127.])
    new_row.append(sb_est_yield)

    # print(f"singleH samples: {[field_names[1], field_names[4]]+ field_names[7:10]}")
    # print(f"nonres samples: {[field_names[2], field_names[3], field_names[6]]}")
    sum_singleH = new_row[1] + new_row[4] + sum(new_row[7:10])
    sum_bkg_no_fit = new_row[2] + new_row[3] + new_row[6]
    new_row.append(new_row[5] / (sum_singleH + sum_bkg_no_fit))
    new_row.append(new_row[5] / (sum_singleH + sb_est_yield))

    table.add_row(new_row)

print(table)


                  ┌──────────────────────────────────────────────────────────┐
[100, 105) 1.162  │███████████████▋                                          │
[105, 110) 4.235  │█████████████████████████████████████████████████████████ │
[110, 115) 1.079  │██████████████▌                                           │
[115, 120) 1.345  │██████████████████▏                                       │
[120, 125) 0      │                                                          │
[125, 130) 0      │                                                          │
[130, 135) 0.8324 │███████████▎                                              │
[135, 140) 1.4    │██████████████████▉                                       │
[140, 145) 0.6207 │████████▍                                                 │
[145, 150) 0.7309 │█████████▉                                                │
[150, 155) 0.8565 │███████████▌                                              │
[155, 160) 0.7134 │█████████▋                       

In [21]:
field_names = ['Category'] + samples + ['nonRes SB fit', 's/b', 's/b with fit']
table = pt.PrettyTable(field_names=field_names, float_format=".3")

not_prev_cut_mask = {}
for name, cuts in cats.items():
    new_row = [name]
    nonRes_sideband = pd.DataFrame({'mass': pd.Series(dtype='float'), 'weight_tot': pd.Series(dtype='float')})

    for sample in samples:
        sample_sum = 0.

        for file in files24:
            if sample not in file: continue

            file_df = pd.read_parquet(file, columns=['mass', 'weight_tot'])
            preds = np.load(file.replace('events.parquet', 'y.npy'))
            pred_fields = ["is_nonRes_bkg_score", "is_Res_bkg_score", "is_ggHH_sig_score", "is_VBFHH_sig_score"]
            for i, field in enumerate(pred_fields):
                file_df[field] = preds[:, i]
                
            if file not in not_prev_cut_mask: not_prev_cut_mask[file] = np.ones(len(file_df), dtype=bool)
            
            pass_cut_mask = not_prev_cut_mask[file]
            for cut_name, cut in cuts.items():
                pass_cut_mask = np.logical_and(
                    pass_cut_mask, file_df.loc[:, f'{cut_name}_score'].gt(cut).to_numpy() if cut_name.endswith('sig') else file_df.loc[:, f'{cut_name}_score'].lt(cut).to_numpy()
                )
            pass_cut_sr_mask = np.logical_and(
                pass_cut_mask,
                np.logical_and(file_df.loc[:, 'mass'].gt(122.5).to_numpy(), file_df.loc[:, 'mass'].lt(127.).to_numpy())
            )

            sample_sum += file_df.loc[pass_cut_sr_mask, 'weight_tot'].sum()

            if sample in ['DDQCDGJET', 'GGJets', 'TTGG']:
                pass_cut_sideband_mask = np.logical_and(
                    pass_cut_mask,
                    np.logical_or(file_df.loc[:, 'mass'].lt(120.).to_numpy(), file_df.loc[:, 'mass'].gt(130.).to_numpy())
                )
                nonRes_sideband = pd.concat([nonRes_sideband, file_df.loc[pass_cut_sideband_mask, ['mass', 'weight_tot']]], ignore_index=True)

            not_prev_cut_mask[file] = np.logical_and(not_prev_cut_mask[file], ~pass_cut_mask)

        new_row.append(sample_sum)

    sb_est_yield = est_yield(nonRes_sideband['mass'], nonRes_sideband['weight_tot'], [100., 180., 5.], [122.5, 127.])
    new_row.append(sb_est_yield)

    # print(f"singleH samples: {[field_names[1], field_names[4]]+ field_names[7:10]}")
    # print(f"nonres samples: {[field_names[2], field_names[3], field_names[6]]}")
    sum_singleH = new_row[1] + new_row[4] + sum(new_row[7:10])
    sum_bkg_no_fit = new_row[2] + new_row[3] + new_row[6]
    new_row.append(new_row[5] / (sum_singleH + sum_bkg_no_fit))
    new_row.append(new_row[5] / (sum_singleH + sb_est_yield))

    table.add_row(new_row)

print(table)


                  ┌──────────────────────────────────────────────────────────┐
[100, 105) 0.6272 │████████████████████████████████▋                         │
[105, 110) 1.096  │█████████████████████████████████████████████████████████ │
[110, 115) 0.7939 │█████████████████████████████████████████▎                │
[115, 120) 0.888  │██████████████████████████████████████████████▏           │
[120, 125) 0      │                                                          │
[125, 130) 0      │                                                          │
[130, 135) 0.3841 │████████████████████                                      │
[135, 140) 0.9361 │████████████████████████████████████████████████▋         │
[140, 145) 0.3134 │████████████████▎                                         │
[145, 150) 0.2935 │███████████████▎                                          │
[150, 155) 0.4451 │███████████████████████▏                                  │
[155, 160) 0.472  │████████████████████████▌        